# Screener.in document downloader - parallel version (Google Colab)

Same downloader as `screener_colab.ipynb`, set up for **several stocks at once** (`--workers`).
Each stock still runs start to finish on one worker; only different stocks run in parallel.

Needs the folder `MyDrive/screener-document-download/` in your Google Drive with `code/` (all `.py` files),
`screener_auth.json` and the `equitydata-screener-all-stocks-*.csv` file.

Run the cells **in order**:
1. **Cell 1** connects Drive and checks the files.
2. **Cell 2** holds the settings (workers, batch, types).
3. **Cell 3** is a safe test (downloads nothing).
4. **Cell 4** is a small real *pilot* batch, **Cell 5** checks it for rate-limit problems.
5. **Cell 6** is the full batch. Change `START` in Cell 2 and re-run it for the next batch.
6. **Cell 7** shows progress. **Cell 8** syncs Drive when you finish.

More workers means more requests per second to Screener and BSE. Start low, check Cell 5, then raise `WORKERS`.
Safe to stop and re-run: finished stocks are listed in `download_tracker.txt` and downloaded files are skipped.

In [ ]:
# Cell 1 - connect Google Drive and check the project files
import os
from google.colab import drive

drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/screener-document-download'  # change if your folder is elsewhere
%cd {PROJECT}

REQUIRED = ['download_screener_docs.py', 'cli.py', 'config.py', 'pipeline.py', 'history.py', 'tracker.py',
            'progress.py', 'http_client.py', 'auth.py', 'html_parser.py', 'excel_export.py',
            'file_downloader.py', 'fs_utils.py', 'log_setup.py', 'models.py', 'symbols_csv.py', 'watchlist.py']
missing = [f'code/{f}' for f in REQUIRED if not os.path.exists(f'code/{f}')]
missing += [f for f in ('screener_auth.json',) if not os.path.exists(f)]
csvs = sorted(f for f in os.listdir('.') if f.startswith('equitydata-screener-all-stocks-') and f.endswith('.csv'))
if missing or not csvs:
    raise FileNotFoundError(f'Missing files: {missing}; stock CSV found: {csvs}. Check the folder in Drive.')
print('OK - project found. Stock list:', csvs[-1])

In [ ]:
# Cell 2 - settings (edit these, then run the cell)
START = 0       # skip this many stocks from the CSV (next batch: 200, 400, ...)
LIMIT = 200     # how many stocks in this batch (5,541 stocks in total)
PILOT = 16      # stocks in the small pilot run (Cell 4)
WORKERS = 4     # stocks downloaded at the same time (1-8). Start with 3-4, raise it only if Cell 5 is clean
TYPES = ['annual', 'transcript', 'ppt']   # remove 'annual' to save Drive space
EXCEL = True    # also click 'Export to Excel' for every stock
FORCE = False   # True = ignore download_tracker.txt and re-check stocks already marked done
MIN_DELAY, MAX_DELAY = 1.0, 2.0   # seconds between requests, per worker (raise both if you see 429s)

args = [f'--types {" ".join(TYPES)}', f'--workers {WORKERS}', f'--min-delay {MIN_DELAY}', f'--max-delay {MAX_DELAY}']
if not EXCEL:
    args.append('--no-excel')
if FORCE:
    args.append('--force')
common_args = ' '.join(args)
print(f'Batch: stocks {START} to {START + LIMIT - 1} | pilot: {PILOT} | workers: {WORKERS} | types: {TYPES}')
print(f'excel: {EXCEL} | force: {FORCE} | delay per worker: {MIN_DELAY}-{MAX_DELAY}s')
print(f'Request rate is about {WORKERS}x a single-worker run.')

In [ ]:
# Cell 3 - TEST: 3 stocks, downloads nothing. Expect 'Found N candidate documents' and 'failed=0'.
# If you see a login/session error, run code/refresh_auth.py or export a fresh screener_auth.json and upload it again.
!python3 code/download_screener_docs.py --all --dry-run --force --start {START} --limit 3 {common_args}

In [ ]:
# Cell 4 - PILOT: a small real download with the parallel workers. Then run Cell 5.
!python3 code/download_screener_docs.py --all --start {START} --limit {PILOT} {common_args}

In [ ]:
# Cell 5 - HEALTH CHECK of the last run: rate limits, blocks, expired session
import glob, re

logs = sorted(glob.glob('logs/run_*.log'))
if not logs:
    print('No log found yet - run Cell 4 first.')
else:
    text = open(logs[-1], encoding='utf-8').read()
    n429 = len(re.findall(r'HTTP 429', text))
    gave_up = len(re.findall(r'Giving up', text))
    forbidden = len(re.findall(r'HTTP 403', text))
    expired = bool(re.search(r'Redirected to login|invalid or expired', text))
    failed = len(re.findall(r'\] Failed ', text))
    print('Log:', logs[-1])
    print(f'HTTP 429 (rate limited): {n429} | gave up after retries: {gave_up} | HTTP 403 (site blocks scripts): {forbidden} | failed files: {failed}')
    if expired:
        print('SESSION EXPIRED: refresh screener_auth.json (code/refresh_auth.py), upload it, then re-run.')
    elif n429 or gave_up > 3:
        print(f'TOO FAST: lower WORKERS (now {WORKERS}) or raise MIN_DELAY / MAX_DELAY in Cell 2, then re-run Cells 2 and 4.')
    else:
        print(f'Looks healthy. You can run the full batch (Cell 6) or try WORKERS = {min(WORKERS + 2, 8)}.')
        print('403s and a few failed files are normal: some company sites block scripts and are retried next run.')

In [ ]:
# Cell 6 - FULL BATCH with the parallel workers. Progress, counts and ETA print below.
# Files go to data/<SYMBOL>/ in your Drive; the log goes to logs/.
# If Colab disconnects: re-run Cells 1, 2 and 6 - finished stocks and files are skipped.
!python3 code/download_screener_docs.py --all --start {START} --limit {LIMIT} {common_args}

In [ ]:
# Cell 7 - STATUS: totals of the last run + what is on disk
summaries = sorted(glob.glob('logs/run_*.summary.json'))
if summaries:
    print('Last run summary:', summaries[-1])
    print(open(summaries[-1]).read())

files = [os.path.join(r, f) for r, _, fs in os.walk('data') for f in fs] if os.path.isdir('data') else []
stocks = [d for d in os.listdir('data') if os.path.isdir(os.path.join('data', d)) and not d.startswith('_')] if files else []
size_gb = sum(os.path.getsize(f) for f in files) / 1024 ** 3
print(f'On Drive: {len(stocks)} stocks, {len(files)} files, {size_gb:.2f} GB (free Drive limit is 15 GB)')

if os.path.exists('download_tracker.txt'):
    done = [l for l in open('download_tracker.txt') if l.strip() and not l.startswith('#')]
    print(f'Tracker: {len(done)} stocks finished with no failures (download_tracker.txt)')

logs = sorted(glob.glob('logs/run_*.log'))
if logs:
    print('\nLast log lines (' + logs[-1] + '):')
    print(''.join(open(logs[-1]).readlines()[-6:]))

In [ ]:
# Cell 8 - when finished: make sure everything is written to Drive
drive.flush_and_unmount()
print('Drive synced.')